# Qualification Agent: pass@k وpass^k
هذا النوتبوك يختبر حساب المقياسين من `evaluation.py` ثم يعرض نتائج التشغيلات المحفوظة.

- `pass@k`: قرار صحيح في مرة واحدة على الأقل من k تشغيلات.
- `pass^k`: قرارات صحيحة في جميع التشغيلات k.
- الحالة غير المكتملة تبقى Pending ولا تدخل في المتوسط.

الاختبارات المحلية لا تقيس دقة النموذج. التقييم الحي اختياري ويستخدم حالات `evaluation_dataset.json`؛ النتائج المتوقعة تحتاج مراجعة بشرية.

In [ ]:
import copy, hashlib, importlib, json, os, sys, unittest
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "agents" / "qualification_agent" / "guardrails.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import Markdown, display
from agents.qualification_agent import evaluation, prompt
importlib.reload(evaluation)
FOLDER = ROOT / "agents" / "qualification_agent"
DATASET_PATH = FOLDER / "evaluation_dataset.json"
RESULTS_PATH = FOLDER / "evaluation_results.json"
dataset = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
print("Dataset:", len(dataset["cases"]), "cases;", sum(c["k"] for c in dataset["cases"]), "planned runs")


## اختبار حساب المقاييس
قرارات مصطنعة معلومة النتيجة لاختبار pass@k وpass^k فقط. التقييم مستقل عن القاردريلز.

In [ ]:
class PassKTests(unittest.TestCase):
    def setUp(self):
        self.case = {"id": "example", "k": 3, "expected": {"decision": "Qualified"}, "evidence": {"value": 1}}

    def runs(self, decisions):
        return [{"decision": d, "seconds": 1, "error": None} for d in decisions]

    def test_all_correct(self):
        score = evaluation.score_case(self.case, self.runs(["Qualified"] * 3))
        self.assertEqual((score["correct"], score["pass_at_k"], score["pass_pow_k"]), (3, True, True))

    def test_one_correct(self):
        score = evaluation.score_case(self.case, self.runs(["Not Qualified", "Qualified", "Needs More Evidence"]))
        self.assertEqual((score["correct"], score["pass_at_k"], score["pass_pow_k"]), (1, True, False))

    def test_none_correct(self):
        score = evaluation.score_case(self.case, self.runs(["Not Qualified"] * 3))
        self.assertEqual((score["pass_at_k"], score["pass_pow_k"]), (False, False))

    def test_incomplete_and_empty_are_pending(self):
        for decisions in ([], ["Qualified"], ["Qualified", "Qualified"]):
            with self.subTest(decisions=decisions):
                score = evaluation.score_case(self.case, self.runs(decisions))
                self.assertFalse(score["complete"])
                self.assertIsNone(score["pass_at_k"])
                self.assertIsNone(score["pass_pow_k"])

    def test_errors_count_as_failure(self):
        runs = self.runs(["Qualified"] * 3)
        runs[0]["error"] = "RuntimeError"
        score = evaluation.score_case(self.case, runs)
        self.assertEqual(score["correct"], 2)
        self.assertTrue(score["pass_at_k"])
        self.assertFalse(score["pass_pow_k"])
        self.assertEqual(score["decisions"][0], "ERROR")

    def test_summary_excludes_pending(self):
        scores = [evaluation.score_case(self.case, self.runs(ds)) for ds in
                  (["Qualified"] * 3, ["Not Qualified"] * 3, ["Qualified"])]
        summary = evaluation.summarize(scores)
        self.assertEqual(summary["completed_cases"], 2)
        self.assertEqual(summary["pass_at_k"], 0.5)
        self.assertEqual(summary["pass_pow_k"], 0.5)
        self.assertIsNone(evaluation.summarize([])["pass_at_k"])

    def test_mixed_k_is_not_averaged(self):
        a = evaluation.score_case(self.case, self.runs(["Qualified"] * 3))
        b = evaluation.score_case(dict(self.case, k=1), self.runs(["Qualified"]))
        self.assertIsNone(evaluation.summarize([a, b])["pass_at_k"])
        self.assertIsNone(evaluation.summarize([a, b])["pass_pow_k"])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(PassKTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "pass@k / pass^k tests failed"


## تشغيل الحالات وحفظ نتائج كل تكرار
`evaluation_dataset.json` هو ملف المدخلات: نمرّر `evidence` من كل حالة إلى إيجنت التأهيل كما هو، k مرات.

`RUN_LIVE = False` يقرأ النتائج المحفوظة فقط. عند اختيار `True` يبدأ تشغيل جديد ويستبدل نتائج التشغيل السابق في `evaluation_results.json`، مع حفظ كل تكرار فور انتهائه:

- `runs[case_id]`: قائمة نتائج تكرارات الحالة، كل تكرار فيه `iteration` و`report` (رد الإيجنت كاملًا) و`decision` و`error`.
- `automated`: نتائج pass@k وpass^k لكل حالة والملخص العام، وتُحدّث بعد كل تكرار.

فشل تكرار يُحفظ كخطأ وتستمر بقية التكرارات. ملف المدخلات لا يتغير. التقارير الكاملة تُحفظ للتشغيلات الجديدة فقط؛ النتائج القديمة التي حُفظت كقرارات لا تحتويها.

In [ ]:
RUN_LIVE = True
if RUN_LIVE:
    from agents.qualification_agent import guardrails, qualification_agent
    # Refresh cached modules so this run uses the current Python files.
    for module in (prompt, guardrails, qualification_agent):
        importlib.reload(module)
    run_qualification_agent = qualification_agent.run_qualification_agent
    print("Agent source:", qualification_agent.__file__)
    dataset = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
    results = {
        "prompt_version": prompt.QUALIFICATION_PROMPT_VERSION,
        "dataset_hash": hashlib.sha256(json.dumps(dataset, sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
        "runs": {},
    }
    for case in dataset["cases"]:
        runs = results["runs"][case["id"]] = []
        for index in range(case["k"]):
            try:
                report = run_qualification_agent(copy.deepcopy(case["evidence"]))
                run = {"iteration": index + 1, "report": report, "decision": report.get("qualification"), "error": None}
            except Exception as exc:
                run = {"iteration": index + 1, "report": None, "decision": None, "error": type(exc).__name__}
            runs.append(run)
            scores = [evaluation.score_case(item, results["runs"].get(item["id"], [])) for item in dataset["cases"]]
            results["automated"] = {"summary": evaluation.summarize(scores), "cases": scores}
            staging = RESULTS_PATH.with_suffix(".tmp")
            staging.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
            staging.replace(RESULTS_PATH)
            print(case["id"], f"{index + 1}/{case['k']}", run["decision"], "error:", run["error"])
else:
    print("Reading saved LLM decisions; no new model calls.")


## تقييم الردود المحفوظة
نقارن قرار كل رد بالقرار المتوقع. الحالات غير المكتملة تبقى Pending. هذه النتائج السابقة منفصلة عن الاختبارات المصطنعة أعلاه.

In [ ]:
dataset = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else {}
expected_hash = hashlib.sha256(json.dumps(dataset, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if results.get("dataset_hash") == expected_hash and results.get("prompt_version") == prompt.QUALIFICATION_PROMPT_VERSION:
    scores = [evaluation.score_case(case, results.get("runs", {}).get(case["id"], [])) for case in dataset["cases"]]
    print(json.dumps(evaluation.summarize(scores), indent=2))
    rows = ["| Case | Runs | Correct | pass@k | pass^k |", "|---|---|---|---|---|"]
    for score in scores:
        at_k, pow_k = ["Pending" if score[key] is None else "Pass" if score[key] else "Fail"
                       for key in ("pass_at_k", "pass_pow_k")]
        rows.append(f"| {score['id']} | {score['runs']}/{score['k']} | {score['correct']} | {at_k} | {pow_k} |")
    display(Markdown("\n".join(rows)))
else:
    print("No matching results: collect responses for the current dataset and prompt first.")
